In [2]:
# Read dataset

import gzip
import json

with gzip.open('../../data/hooktheory/Hooktheory.json.gz', 'r') as f:
    dataset = json.load(f)

print(len(dataset))

26175


In [46]:
dataset_split_all = {}
for split in ["train", "valid", "test"]:
    dataset_split = {
        k: v
        for k, v in dataset.items()
        if v["split"] == split.upper()
        and "MELODY" in v["tags"]
        and "HARMONY" in v["tags"]
        and "TEMPO_CHANGES" not in v["tags"]
    }
    dataset_split = list(dataset_split.values())
    dataset_split_all[split] = dataset_split

train_set = dataset_split_all['train']
print(len(train_set))

19052


In [113]:
# Inspect one

example = train_set[14714]
print(json.dumps(example, indent=2))

{
  "tags": [
    "NO_SWING",
    "AUDIO_AVAILABLE",
    "USER_ALIGNMENT",
    "REFINED_ALIGNMENT",
    "MELODY",
    "HARMONY"
  ],
  "split": "TRAIN",
  "hooktheory": {
    "id": "yvmrqAZdoOW",
    "artist": "beatles",
    "song": "hey-jude",
    "annotators": [
      "coguma",
      "nathanwomack",
      "EMANT",
      "Vaz123"
    ],
    "urls": {
      "artist": "https://www.hooktheory.com/theorytab/artists/b/beatles",
      "song": "https://www.hooktheory.com/theorytab/view/beatles/hey-jude",
      "clip": "https://hookpad.hooktheory.com/?idOfSong=yvmrqAZdoOW"
    }
  },
  "youtube": {
    "id": "A_MjCqQoLLA",
    "url": "https://www.youtube.com/watch?v=A_MjCqQoLLA",
    "duration": 489.52308390022677
  },
  "alignment": {
    "swing": "STRAIGHT",
    "user": {
      "beats": [
        0,
        32
      ],
      "times": [
        57.13491184176701,
        82.66710682374341
      ]
    },
    "refined": {
      "beats": [
        0,
        1,
        2,
        3,
        4,


In [128]:
# Parse alignment

import numpy as np
from scipy.interpolate import interp1d

beat_to_time_fn = interp1d(
    example['alignment']['refined']['beats'],
    example['alignment']['refined']['times'],
    kind='linear',
    fill_value='extrapolate')
start_time = beat_to_time_fn(0)
num_beats = example['annotations']['num_beats']
end_time = beat_to_time_fn(num_beats)
print(start_time, end_time)

for i in range(32):
    print(beat_to_time_fn(i))

57.1991223680828 82.76912236808279
57.1991223680828
58.00912236808279
58.82912236808279
59.6291223680828
60.429122368082794
61.2291223680828
62.03912236808279
62.82912236808279
63.65912236808279
64.48912236808279
65.2891223680828
66.0891223680828
66.90912236808279
67.68912236808279
68.4891223680828
69.2491223680828
70.04912236808279
70.8691223680828
71.6691223680828
72.4691223680828
73.2491223680828
74.0591223680828
74.8391223680828
75.6191223680828
76.4191223680828
77.2191223680828
77.9991223680828
78.79912236808279
79.57912236808279
80.3691223680828
81.1691223680828
81.9691223680828


In [129]:
# Interpret annotation as MIDI

CHORD_OCTAVE = 4
MELODY_OCTAVE = 4

import pretty_midi

midi = pretty_midi.PrettyMIDI()

# Create click track
# click = pretty_midi.Instrument(program=0, is_drum=True)
# midi.instruments.append(click)
# beats_per_bar = example['annotations']['meters'][0]['beats_per_bar']
# for b in range(num_beats + 1):
#     downbeat = b % beats_per_bar == 0
#     click.notes.append(pretty_midi.Note(
#         100 if downbeat else 75, 
#         37 if downbeat else 31,
#         beat_to_time_fn(b),
#         beat_to_time_fn(b + 1)))

# Create harmony track
harmony = pretty_midi.Instrument(program=0)
midi.instruments.append(harmony)
for c in example['annotations']['harmony']:
    root_position_pitches = [c['root_pitch_class']]
    for interval in c['root_position_intervals']:
        root_position_pitches.append(root_position_pitches[-1] + interval)
    for p in root_position_pitches:
        harmony.notes.append(pretty_midi.Note(
            67,
            p + CHORD_OCTAVE * 12,
            beat_to_time_fn(c['onset']),
            beat_to_time_fn(c['offset'])
        ))

# Create melody track
melody = pretty_midi.Instrument(program=0)
midi.instruments.append(melody)
for n in example['annotations']['melody']:
    melody.notes.append(pretty_midi.Note(
        100,
        n['pitch_class'] + (MELODY_OCTAVE + n['octave']) * 12,
        beat_to_time_fn(n['onset']),
        beat_to_time_fn(n['offset'])
    ))

midi.write(f"midis/annotations_{example['hooktheory']['song']}_{example['hooktheory']['id']}.midi")

In [84]:
# Retrieve and decode audio (this one is CC-licensed)
# Thanks JohnJRenns https://coolandnewwebcomic.bandcamp.com/track/plague !

import librosa
from IPython.display import display, Audio

!youtube-dl --format bestaudio/best --output 'audio' {example['youtube']['url']}
audio, sr = librosa.load('audio', sr=None, mono=True, offset=start_time, duration=end_time - start_time)

/bin/bash: line 1: youtube-dl: command not found


/tmp/ipykernel_191805/1801943002.py:8: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load('audio', sr=None, mono=True, offset=start_time, duration=end_time - start_time)
/home/abel/Projects/Thesis/realchords-pytorch/.venv/lib64/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: 'audio'

In [9]:
# Synthesize aligned preview

annotations_audio = midi.fluidsynth(fs=sr)
annotations_audio = annotations_audio[round(start_time * sr):]
annotations_audio = annotations_audio[:audio.shape[0]]
if annotations_audio.shape[0] < audio.shape[0]:
    annotations_audio = np.pad(annotations_audio, [(0, audio.shape[0] - annotations_audio.shape[0])])
display(Audio([audio, annotations_audio], rate=sr))

NameError: name 'sr' is not defined